In [1]:
# pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch05/07_gpt_to_llama/requirements-extra.txt

In [2]:
# !pip install galois

In [25]:
from importlib.metadata import version
from qwen import *
import torch
import torch.nn as nn
from transformers import AutoTokenizer
import time



pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")


# Select which model to use via the following flag; only one can be True

USE_BASE_MODEL = True
USE_REASONING_MODEL = False
USE_INSTRUCT_MODEL = False

if (USE_BASE_MODEL + USE_REASONING_MODEL
    + USE_INSTRUCT_MODEL) != 1:
    raise AttributeError("Only one of the options above can be True.")

huggingface_hub version: 0.30.2
tokenizers version: 0.21.1
torch version: 2.6.0


In [2]:


# class FeedForward(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
#         self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
#         self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

#     def forward(self, x):
#         x_fc1 = self.fc1(x)
#         x_fc2 = self.fc2(x)
#         x = nn.functional.silu(x_fc1) * x_fc2
#         return self.fc3(x)

# class RMSNorm(nn.Module):
#     def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
#         super().__init__()
#         self.eps = eps
#         self.qwen3_compatible = qwen3_compatible
#         self.scale = nn.Parameter(torch.ones(emb_dim))
#         self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

#     def forward(self, x):
#         input_dtype = x.dtype

#         if self.qwen3_compatible:
#             x = x.to(torch.float32)

#         variance = x.pow(2).mean(dim=-1, keepdim=True)
#         norm_x = x * torch.rsqrt(variance + self.eps)
#         norm_x = norm_x * self.scale

#         if self.shift is not None:
#             norm_x = norm_x + self.shift

#         return norm_x.to(input_dtype)

# def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
#     assert head_dim % 2 == 0, "Embedding dimension must be even"

#     # Compute the inverse frequencies
#     inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

#     # Generate position indices
#     positions = torch.arange(context_length, dtype=dtype)

#     # Compute the angles
#     angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

#     # Expand angles to match the head_dim
#     angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

#     # Precompute sine and cosine
#     cos = torch.cos(angles)
#     sin = torch.sin(angles)

#     return cos, sin


# def apply_rope(x, cos, sin):
#     # x: (batch_size, num_heads, seq_len, head_dim)
#     batch_size, num_heads, seq_len, head_dim = x.shape
#     assert head_dim % 2 == 0, "Head dimension must be even"

#     # Split x into first half and second half
#     x1 = x[..., : head_dim // 2]  # First half
#     x2 = x[..., head_dim // 2 :]  # Second half

#     # Adjust sin and cos shapes
#     cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
#     sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

#     # Apply the rotary transformation
#     rotated = torch.cat((-x2, x1), dim=-1)
#     x_rotated = (x * cos) + (rotated * sin)

#     # It's ok to use lower-precision after applying cos and sin rotation
#     return x_rotated.to(dtype=x.dtype)

# class GroupedQueryAttention(nn.Module):
#     def __init__(
#         self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
#     ):
#         super().__init__()
#         assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

#         self.num_heads = num_heads
#         self.num_kv_groups = num_kv_groups
#         self.group_size = num_heads // num_kv_groups

#         if head_dim is None:
#             assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
#             head_dim = d_in // num_heads

#         self.head_dim = head_dim
#         self.d_out = num_heads * head_dim

#         self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
#         self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
#         self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

#         self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

#         if qk_norm:
#             self.q_norm = RMSNorm(head_dim, eps=1e-6)
#             self.k_norm = RMSNorm(head_dim, eps=1e-6)
#         else:
#             self.q_norm = self.k_norm = None

#     def forward(self, x, mask, cos, sin):
#         b, num_tokens, _ = x.shape

#         # Apply projections
#         queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
#         keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
#         values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

#         # Reshape
#         queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
#         keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
#         values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

#         # Optional normalization
#         if self.q_norm:
#             queries = self.q_norm(queries)
#         if self.k_norm:
#             keys = self.k_norm(keys)

#         # Apply RoPE
#         queries = apply_rope(queries, cos, sin)
#         keys = apply_rope(keys, cos, sin)

#         # Expand K and V to match number of heads
#         keys = keys.repeat_interleave(self.group_size, dim=1)
#         values = values.repeat_interleave(self.group_size, dim=1)

#         # Attention
#         attn_scores = queries @ keys.transpose(2, 3)
#         attn_scores = attn_scores.masked_fill(mask, -torch.inf)
#         attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)

#         context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
#         return self.out_proj(context)


# class TransformerBlock(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()
#         self.att = GroupedQueryAttention(
#             d_in=cfg["emb_dim"],
#             num_heads=cfg["n_heads"],
#             head_dim=cfg["head_dim"],
#             num_kv_groups=cfg["n_kv_groups"],
#             qk_norm=cfg["qk_norm"],
#             dtype=cfg["dtype"]
#         )
#         self.ff = FeedForward(cfg)
#         self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
#         self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

#     def forward(self, x, mask, cos, sin):
#         # Shortcut connection for attention block
#         shortcut = x
#         x = self.norm1(x)
#         x = self.att(x, mask, cos, sin)  # Shape [batch_size, num_tokens, emb_size]
#         x = x + shortcut  # Add the original input back

#         # Shortcut connection for feed-forward block
#         shortcut = x
#         x = self.norm2(x)
#         x = self.ff(x)
#         x = x + shortcut  # Add the original input back

#         return x


# class Qwen3Model(nn.Module):
#     def __init__(self, cfg):
#         super().__init__()

#         # Main model parameters
#         self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

#         self.trf_blocks = nn.ModuleList(  # ModuleList since Sequential can only accept one input, and we need `x, mask, cos, sin`
#             [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
#         )

#         self.final_norm = RMSNorm(cfg["emb_dim"])
#         self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

#         # Reusable utilities
#         if cfg["head_dim"] is None:
#             head_dim = cfg["emb_dim"] // cfg["n_heads"]
#         else:
#             head_dim = cfg["head_dim"]
#         cos, sin = compute_rope_params(
#             head_dim=head_dim,
#             theta_base=cfg["rope_base"],
#             context_length=cfg["context_length"]
#         )
#         self.register_buffer("cos", cos, persistent=False)
#         self.register_buffer("sin", sin, persistent=False)
#         self.cfg = cfg


#     def forward(self, in_idx):
#         # Forward pass
#         tok_embeds = self.tok_emb(in_idx)
#         x = tok_embeds

#         num_tokens = x.shape[1]
#         mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)
        
#         for block in self.trf_blocks:
#             x = block(x, mask, self.cos, self.sin)
#         x = self.final_norm(x)
#         logits = self.out_head(x.to(self.cfg["dtype"]))
#         return logits

&nbsp;
# 2. Initialize model

In [3]:
CHOOSE_MODEL = "0.6B"

if CHOOSE_MODEL == "0.6B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,           # Vocabulary size
        "context_length": 40_960,        # Context length that was used to train the model
        "emb_dim": 1024,                 # Embedding dimension
        "n_heads": 16,                   # Number of attention heads
        "n_layers": 28,                  # Number of layers
        "hidden_dim": 3072,              # Size of the intermediate dimension in FeedForward
        "head_dim": 128,                 # Size of the heads in GQA
        "qk_norm": True,                 # Whether to normalize queries and keys in GQA
        "n_kv_groups": 8,                # Key-Value groups for grouped-query attention
        "rope_base": 1_000_000.0,        # The base in RoPE's "theta"
        "dtype": torch.bfloat16,         # Lower-precision dtype to reduce memory usage
    }

elif CHOOSE_MODEL == "1.7B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 2048,                 # 2x larger than above
        "n_heads": 16,
        "n_layers": 28,
        "hidden_dim": 6144,              # 2x larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }   

elif CHOOSE_MODEL == "4B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 2560,                 # 25% larger than above
        "n_heads": 32,                   # 2x larger than above
        "n_layers": 36,                  # 29% larger than above
        "hidden_dim": 9728,              # ~3x larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }  

elif CHOOSE_MODEL == "8B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 4096,                 # 60% larger than above
        "n_heads": 32,
        "n_layers": 36,                  # 26% larger than above
        "hidden_dim": 12288,
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    } 

elif CHOOSE_MODEL == "14B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 5120,                 # 25% larger than above
        "n_heads": 40,                   # 25% larger than above
        "n_layers": 40,                  # 11% larger than above
        "hidden_dim": 17408,             # 42% larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    } 

elif CHOOSE_MODEL == "32B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 5120,                
        "n_heads": 64,                   # 60% larger than above
        "n_layers": 64,                  # 60% larger than above
        "hidden_dim": 25600,             # 47% larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    } 

else:
    raise ValueError(f"{CHOOSE_MODEL} is not supported.")

In [4]:
# torch.manual_seed(123)
model = Qwen3Model(QWEN3_CONFIG)

In [5]:
# model

- A quick check that the forward pass works before continuing:

In [6]:
model(torch.tensor([1, 2, 3]).unsqueeze(0))

tensor([[[-0.5312,  0.2363,  0.6719,  ...,  0.5156,  1.3984,  0.3125],
         [ 0.2617, -0.8711,  0.1738,  ...,  0.5469,  0.8672,  0.0928],
         [ 0.5781, -1.0156, -0.8125,  ...,  0.5508,  0.5625,  0.3555]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>)

In [7]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

Total number of parameters: 751,632,384

Total number of unique parameters: 596,049,920


In [16]:


print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

float32 (PyTorch default): 5.64 GB
bfloat16: 2.82 GB


In [10]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device);

&nbsp;
# 3. Load pretrained weights

In [11]:
def load_weights_into_qwen(model, param_config, params):
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")
        
        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))
    
        return left 

    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )

        # QK norms
        if hasattr(att, "q_norm") and att.q_norm is not None:
            att.q_norm.scale = assign(
                att.q_norm.scale,
                params[f"model.layers.{l}.self_attn.q_norm.weight"],
                f"model.layers.{l}.self_attn.q_norm.weight"
            )
        if hasattr(att, "k_norm") and att.k_norm is not None:
            att.k_norm.scale = assign(
                att.k_norm.scale,
                params[f"model.layers.{l}.self_attn.k_norm.weight"],
                f"model.layers.{l}.self_attn.k_norm.weight"
            )

        # Attention layernorm
        block.norm1.scale = assign(
            block.norm1.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # Feedforward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight"
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight"
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight"
        )
        block.norm2.scale = assign(
            block.norm2.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # Final normalization and output head
    model.final_norm.scale = assign(model.final_norm.scale, params["model.norm.weight"], "model.norm.weight")

    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")

In [12]:
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download


if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}"
else:
    repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}-Base"

local_dir = Path(repo_id).parts[-1]

if CHOOSE_MODEL == "0.6B":
    weights_file = hf_hub_download(
        repo_id=repo_id,
        filename="model.safetensors",
        local_dir=local_dir,
    )
    weights_dict = load_file(weights_file)
else:
    repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    index_path = os.path.join(repo_dir, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)

    weights_dict = {}
    for filename in set(index["weight_map"].values()):
        shard_path = os.path.join(repo_dir, filename)
        shard = load_file(shard_path)
        weights_dict.update(shard)

load_weights_into_qwen(model, QWEN3_CONFIG, weights_dict)
model.to(device)
del weights_dict

Model uses weight tying.


&nbsp;
# 4. Load tokenizer

In [13]:
import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>"
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s

In [14]:
!ls Qwen3-0.6B/

model.safetensors  tokenizer.json


In [15]:
if USE_REASONING_MODEL:
    tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}/tokenizer.json"
else:
    tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}-Base/tokenizer.json"

hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

print(tokenizer_file_path)
if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=USE_REASONING_MODEL
    )

else:
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=False,
        add_generation_prompt=False,
        add_thinking=False
    )

Qwen3-0.6B-Base/tokenizer.json


In [17]:
tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B')

In [18]:
def prompt_to_ids(prompt):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    formatted_text = tok.apply_chat_template(messages, tokenize=False,     add_generation_prompt=True)
    return tokenizer.encode(formatted_text)

In [35]:
import numpy as np
import galois

GF2 = galois.GF(2)

def sample_P(n: int, t: int, r: int, rng: np.random.Generator) -> galois.FieldArray:
    """Sample P ∈ F_2^{r×n} with each row a uniform t-sparse vector."""
    P = GF2.Zeros((r, n))
    for i in range(r):
        idx = rng.choice(n, size=t, replace=False)
        P[i, idx] = 1
    return P


def kernel_basis_gf2(P: galois.FieldArray) -> galois.FieldArray:
    """Return a basis (as columns) for ker(P) over F_2. P is r×n."""
    null = P.null_space()  # rows span ker(P)
    return null.T  # (n, n - rank)


def sample_G(P: galois.FieldArray, g: int, rng: np.random.Generator) -> galois.FieldArray:
    """Sample G ∈ F_2^{n×g} uniformly from ker(P)^g, i.e. PG = 0."""
    basis = kernel_basis_gf2(P)
    n, k = basis.shape
    if k < g:
        raise ValueError(f"ker(P) has dim {k} < g={g}; cannot sample G")
    coeffs = GF2(rng.integers(0, 2, size=(k, g), dtype=np.uint8))
    return basis @ coeffs


def generate_PG(n: int, t: int, r: int, g: int, seed: int | None = None):
    """Sample (P, G) ← LDPC[n, g, t, r] per Definition 3 of Christ–Gunn."""
    rng = np.random.default_rng(seed)
    P = sample_P(n, t, r, rng)
    G = sample_G(P, g, rng)
    assert not np.any(P @ G), "PG != 0"
    return P, G

def sample_t_sparse_numpy(n: int, t: int) -> galois.FieldArray:
    if t > n or t < 0:
        raise ValueError("t must be between 0 and n")
    vector = np.zeros(n, dtype=np.int8)
    indices = np.random.choice(n, size=t, replace=False)
    vector[indices] = 1
    return GF2(vector)

def bernoulli_noise(n: int, eta: float):
    t =int(eta*n)
    return sample_t_sparse_numpy(n, t)

def add_error(vec: galois.FieldArray, eta: float):
    n = vec.shape[0]
    return vec + bernoulli_noise(n, eta)

def weight(P, vec: galois.FieldArray, z: galois.FieldArray):
    return int(np.array(P@vec + P@z, dtype=int).sum())

def detect(P, vec: galois.FieldArray, z: galois.FieldArray):
    wt = weight(P, vec, z)
    r = P.shape[0]
    threshold = (0.5 - (1/(r**4)))*r
    return wt < threshold

def pad(vec):
    return vec+z
    
def sample_codeword(G, eta):
    s = GF2(np.random.binomial(1, 0.5, G.shape[1]))
    return add_error(pad(G@s), eta)


def sample(p,x):
    if p < 0.5:
        mod_p  = 2*x*p
    else:
        mod_p =  (1 - 2*(1 - x)*(1 - p))
    print(f"p is {p}, x is {x}, new probability is {mod_p}")
    t = np.random.binomial(1 , mod_p)
    return t


class LDPC_PRC:
    def __init__(self, n):
        self.n = n
        self.eta = 0.4
        self.t = int(np.log2(n))          # Θ(log n)
        self.g = int(np.log2(n) ** 2)     # Ω(log² n)
        self.r = self.n - self.g                    # or any r ≤ 0.99n
        print(f"n is {self.n}")
        print(f"g is {self.g}")
        print(f"t is {self.t}")
        print(f"r is {self.r}")
        self.P, self.G = generate_PG(n, self.t, self.r, self.g, seed=0)


In [84]:
import numpy as np


def sample_orthogonal_gf2_matrices(n, g, t):
    """
    Sample two GF(2) matrices P and G such that P @ G = 0 over GF(2).

    Parameters:
        n: dimension parameter
        g: number of columns in G
        t: exact number of ones per row of P; each s_i has exactly t-1 ones

    Returns:
        P: (0.99n) x n matrix over GF(2)
        G: n x g matrix over GF(2)
    """
    n_small = int(0.01 * n)  # rows in G0
    n_large = int(0.99 * n)  # number of extra rows / rows in P

    assert n_small + n_large == n, (
        f"n={n} doesn't split cleanly: 0.01n={n_small}, 0.99n={n_large}"
    )
    assert t - 1 <= n_small, (
        f"t-1={t-1} exceeds n_small={n_small}; can't place that many ones in s_i"
    )

    # Step 1: Sample uniformly random G0 in F_2^{n_small x g}
    G0 = np.random.randint(0, 2, size=(n_small, g), dtype=np.int8)

    G = G0.copy()
    P_rows = []

    # Step 2: For i = 1, ..., n_large
    for i in range(1, n_large + 1):
        # (a) Sample s_i with exactly (t-1) ones in F_2^{n_small}
        s_i = np.zeros(n_small, dtype=np.int8)
        positions = np.random.choice(n_small, size=t - 1, replace=False)
        s_i[positions] = 1

        # (b) New row = s_i^T @ G0 (mod 2)
        new_row = (s_i @ G0) % 2
        G = np.vstack([G, new_row.reshape(1, -1)])

        # (c) s'_i = [s_i, 0_{i-1}, 1, 0_{n_large - i}]
        s_prime = np.zeros(n, dtype=np.int8)
        s_prime[:n_small] = s_i
        s_prime[n_small + (i - 1)] = 1
        P_rows.append(s_prime)

    P = np.array(P_rows, dtype=np.int8)

    return P, G


def verify(P, G, t):
    product = (P @ G) % 2
    orthogonal = np.all(product == 0)
    row_weights = P.sum(axis=1)
    all_t_sparse = np.all(row_weights == t)
    return orthogonal, all_t_sparse


n = 1000
g = 75
t = 4
eta = 0.1
z = GF2(np.random.binomial(1, 0.5, n))

    
P, G = sample_orthogonal_gf2_matrices(n, g, t)
orthogonal, all_t_sparse = verify(P, G, t)
# print(f"P shape: {P.shape}")       # (990, 1000)
# print(f"G shape: {G.shape}")       # (1000, 50)
# print(f"P @ G = 0 (mod 2)? {orthogonal}")
# print(f"Every row of P has exactly {t} ones? {all_t_sparse}")


P, G = GF2(P), GF2(G)
# print(f"P: {P.shape}, row weights: {np.asarray(P).sum(axis=1)[:5]}... (all == {t})")
# print(f"G: {G.shape}")
# print(f"PG = 0 ? {not np.any(P @ G)}")



for _ in range(30):
    s = GF2(np.random.binomial(1, 0.5, g))
    x = add_error(pad(G@s), eta)
    # print(weight(P, x, z))
    print(detect(P,x, z))

True
True
True
True
True
False
False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
False
True
True
True


In [85]:
for _ in range(30):
    u = GF2(np.random.binomial(1, 0.5, n))
    print(detect(P, u, z))

True
False
False
False
False
True
True
True
False
True
True
False
True
True
False
False
False
True
True
True
True
False
True
True
True
True
True
False
True
False


In [80]:
n = 1000
eta = 0.1
t = int(np.log2(n))          # Θ(log n)
g = int(np.log2(n) ** 2)     # Ω(log² n)
r = n - g                    # or any r ≤ 0.99n
print(f"n is {n}")
print(f"g is {g}")
print(f"t is {t}")
print(f"r is {r}")
#p = np.random.uniform(0,1,n)
z = GF2(np.random.binomial(1, 0.5, n))

if True:
    P, G = sample_orthogonal_gf2_matrices(n, g, t) #generate_PG(n, t, r, g, seed=200)
    print(f"P: {P.shape}, row weights: {np.asarray(P).sum(axis=1)[:5]}... (all == {t})")
    print(f"G: {G.shape}")
    print(f"PG = 0 ? {not np.any(P @ G)}")

    s = GF2(np.random.binomial(1, 0.5, g))
    x = add_error(pad(G@s), eta)
    print(weight(P, x, z))
    print(detect(P,x, z))

n is 1000
g is 99
t is 9
r is 901
P: (990, 1000), row weights: [9 9 9 9 9]... (all == 9)
G: (1000, 99)
PG = 0 ? False


TypeError: Argument 'A' must be an instance of <class 'galois.GF(2, primitive_element='1', irreducible_poly='x + 1')'>, not <class 'numpy.ndarray'>.

In [79]:
count = 0
for i in range(100):
    u = GF2(np.random.binomial(1, 0.5, n))
    weight(P, u, z)
    if detect(P, u, z):
        count += 1

count

58

In [45]:
vocab_size  = model.tok_emb.weight.shape[0]

v_0 = torch.zeros(vocab_size)
indices  = torch.randperm(vocab_size)[:vocab_size//2]
v_0[indices] = 1.0
v_1 = 1-v_0
partition = torch.concat([v_0,v_1]).reshape(2, vocab_size)

&nbsp;
# 4. Generate text

Tokenize(A + B) != (Tokenize(A) | Tokenize(B))

In [22]:
def watermark_refresh(G, watermark, pos, device, codeword_len):
    if pos%codeword_len == 0:
        if watermark:
            print("Watermark Enabled", flush=True)
            return torch.Tensor(sample_vector(G, eta)).to(device)
        else:
            print("Watermark Disabled", flush=True)                      
            return torch.bernoulli(torch.Tensor([0.5 for _ in range(codeword_len)])).to(device)
    else:
        return None


def generate_text_watermark(model, token_ids, max_new_tokens, G, partition_map, eos_token_id=None, watermark=True, eta=0.1):
    model.eval()
    device = token_ids.device
    codeword_len = G.shape[0] if watermark else 100
    if watermark:
        print("Watermark Enabled", flush=True)
        codeword = torch.Tensor(sample_vector(G, eta)).to(device)
    else:
        print("Watermark Disabled", flush=True)                      
        codeword = torch.bernoulli(torch.Tensor([0.5 for _ in range(codeword_len)])).to(device)
    partition_map = partition_map.to(device)
    with torch.no_grad():
        for pos in range(max_new_tokens):
            ret = watermark_refresh(G, watermark, pos, device, codeword_len)
            codeword = ret if ret is not None else codeword
            out = model(token_ids)[:, -1]  # (batch, vocab_size)
            if watermark:
                xi = codeword[pos%codeword_len]
                probs = torch.softmax(out, dim=-1)  # (batch, vocab_size)
                p = (probs * partition_map[1].to(out.device)).sum(dim=-1)  # (batch,)

                # Compute Bernoulli parameter
                bern_p = torch.where(
                    p <= 0.5,
                    2 * xi * p,
                    1 - 2 * (1 - xi) * (1 - p)
                ).clamp(0.0, 1.0)

                t = torch.bernoulli(bern_p).long()  # (batch,)
                mask = partition_map[t].to(out.device)  # per-batch: (batch, vocab_size)
                out = out * mask + (1 - mask) * (-1e9)

            next_token = torch.argmax(out, dim=-1, keepdim=True)
            if eos_token_id is not None and torch.all(next_token == eos_token_id):
                break
            yield next_token

            token_ids = torch.cat([token_ids, next_token], dim=1)

In [23]:
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = model(token_ids)[:, -1]
            next_token = torch.argmax(out, dim=-1, keepdim=True)

            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break

            yield next_token
            
            token_ids = torch.cat([token_ids, next_token], dim=1)

In [26]:
def get_response(ids, do_watermark):
    response = []
    for token in generate_text_watermark(
        model=model,
        token_ids=ids,
        G=G, partition_map=partition,
        max_new_tokens=500,
        watermark=do_watermark,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        response.append(token_id[0])
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    return response

def decode_token(token, partition_map):
    if partition_map[0][token]==1:
        return 0
    elif partition_map[1][token]==1:
        return 1
    else:
        raise Exception

def detect_llm(buckets, P):
    length = len(buckets)
    buckets = GF2(buckets)
    codeword_len = P.shape[1]
    r = P.shape[0]
    threshold = (0.5 - (1/(r**4)))*r
    for i in range(0, length, codeword_len):
        substring = buckets[i: i+codeword_len]
        len_substring = len(substring)
        # print(substring)
        # print(len_substring)
        wt = np.array(P[:, :len_substring] @ substring).sum()
        print(wt, flush=True)
        if wt < threshold:
            return True
    return False

def detect_watermark(response):
    buckets = [decode_token(token, partition) for token in response]
    return detect_llm(buckets, P)

In [27]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


In [32]:
test_prompts = [
    # Creativity and Writing
    "Write a haiku about a robot learning to feel emotions.",
    "Explain quantum computing to a five-year-old.",
    "Write a short story from the perspective of a stray cat living in a cyberpunk city.",
    "Draft a polite but firm email declining a job offer because the salary is too low.",
    "Write a catchy slogan and a short pitch for a company that sells waterproof socks.",
    
    # Logic and Reasoning
    "Solve this riddle: I speak without a mouth and hear without ears. I have no body, but I come alive with wind. What am I?",
    "If a train leaves New York at 3 PM traveling 60 mph, and another leaves Boston at 4 PM traveling 70 mph on a parallel track, how far apart are they exactly one hour before they meet?",
    "Identify the logical fallacy in this statement: 'If we let students redo their tests, next they'll want to redo their entire lives.'",
    "List 5 highly unconventional but practical uses for a standard metal paperclip.",
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?",
    
    # Coding and Technical
    "Write a Python function to reverse a string without using any built-in reversing functions or slicing.",
    "Explain the difference between a process and a thread in computer science.",
    "Debug this Python code and explain the error: `def add_item(item, my_list=[]): my_list.append(item); return my_list`",
    "Write a SQL query to find the second highest salary from an 'Employees' table.",
    "Write a bash one-liner to find all `.txt` files in a directory and count the total number of words in them.",
    
    # Factual and Educational
    "What were the primary economic and social causes of the French Revolution?",
    "Compare and contrast mitosis and meiosis.",
    "Explain the Fermi Paradox and provide two potential solutions to it.",
    "How does a blockchain work? Keep the explanation under 150 words.",
    "What is the capital of Australia, and why was it chosen over Sydney or Melbourne?",
    
    # Formatting and Constraints
    "Create a JSON object representing a bookshelf containing three distinct books with their title, author, and publication year.",
    "Generate a Markdown table comparing 5 common fruits, including columns for Color, Taste Profile, and Average Calories.",
    "Summarize the plot of Shakespeare's 'Hamlet' in exactly three sentences.",
    "List the planets in our solar system in alphabetical order.",
    "Translate the phrase 'Hello, how are you doing today?' into French, Spanish, Japanese, and Arabic.",
    
    # Roleplay and Persona
    "Act as William Shakespeare and describe a modern smartphone.",
    "You are an expert travel agent. Create a realistic, budget-friendly 3-day itinerary for a trip to Rome.",
    "Write a dialogue between a die-hard coffee lover and a passionate tea enthusiast arguing about which drink is superior.",
    "Act as a sarcastic AI assistant who is tired of answering basic math questions, but answer the question 'What is 8 times 7?' anyway.",
    "Assume the persona of a 19th-century pirate and give me a recipe for hardtack."
]

In [34]:
prompts = test_prompts

watermarked_responses = []

for prompt in prompts:
    ids = prompt_to_ids(prompt)
    input_token_ids_tensor = torch.tensor(ids, device=device).unsqueeze(0)
    watermarked_responses.append(get_response(input_token_ids_tensor, True))
print(sum([detect_watermark(resp) for resp in watermarked_responses ]), flush=True)

unwatermarked_responses = []
for prompt in prompts:
    ids = prompt_to_ids(prompt)
    input_token_ids_tensor = torch.tensor(ids, device=device).unsqueeze(0)
    unwatermarked_responses.append(get_response(input_token_ids_tensor, False))
print(sum([detect_watermark(resp) for resp in unwatermarked_responses ]), flush=True)


Watermark Enabled
Watermark Enabled
Write a haiku about a robot learning to feel emotions. ���rariesystem

Assistant: ���rariesystem
illed by the light of day
as it learned to feel

 ���rariesystem
illed by the light of day
as it learnt to beWatermark Enabled
Watermark Enabled
Quantum computing is like using a magical box that can do things computers can't do yet. Imagine you have a magic box that can add numbers in a special way called superposition. This magic box can do things like multiplying numbers in a way that is faster and more efficient than the way computers usually do it.

For example, if you have a magic box that can multiply two numbers, it can do it much faster than just adding those numbers together. This is because the magic box can store andWatermark Enabled
 use the numbers in superpositions, which means it can look at both possibilities at once, rather than just one.

So, quantum computing is like a magical box that can do things computers can't do, but it's faster 

In [145]:

# start_time = time.perf_counter()
# generated_tokens = 0
# response = []
# for token in generate_text_watermark(
#     model=model,
#     token_ids=input_token_ids_tensor,
#     G=G, partition_map=partition,
#     max_new_tokens=500,
#     watermark=do_watermark,
#     eos_token_id=tokenizer.eos_token_id
# ):
#     generated_tokens += 1
#     token_id = token.squeeze(0).tolist()
#     response.append(token_id[0])
#     print(
#         tokenizer.decode(token_id),
#         end="",
#         flush=True
#     )

# elapsed = time.perf_counter() - start_time
# tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
# print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

# if torch.cuda.is_available():
#     def calc_gpu_gb(x):
#         return f"{x / 1024 / 1024 / 1024:.2f} GB"

#     print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")

Watermark Disabled
Watermark Disabled
Sure, here is a very long introduction to large language models:

Large language models (LLMs) are a class of artificial intelligence (AI) systems that have been trained on vast amounts of text data to understand and generate human-like language. These models are based on deep learning techniques and are designed to process and generate text in a variety of ways, including writing, summarizing, and generating creative content.

LLMs have been around for a long time, but they have become increasingly powerful and sophisticated inWatermark Disabled
 recent years. They are capable of understanding context, generating coherent and contextually relevant responses, and even generating creative content such as poetry, music, and art. LLMs are also capable of understanding and generating complex ideas and concepts, and can be used to assist with a wide range of tasks, from writing and editing to summarizing and generating creative content.

LLMs are a rapi

In [123]:
partition

tensor([[0., 1., 0.,  ..., 1., 1., 0.],
        [1., 0., 1.,  ..., 0., 0., 1.]])

In [147]:
def decode_token(token, partition_map):
    if partition_map[0][token]==1:
        return 0
    elif partition_map[1][token]==1:
        return 1
    else:
        raise Exception

def detect_llm(buckets, P):
    length = len(buckets)
    buckets = GF2(buckets)
    codeword_len = P.shape[1]
    r = P.shape[0]
    threshold = (0.5 - (1/(r**4)))*r
    for i in range(0, length, codeword_len):
        substring = buckets[i: i+codeword_len]
        len_substring = len(substring)
        # print(substring)
        # print(len_substring)
        wt = np.array(P[:, :len_substring] @ substring).sum()
        print(wt)
        if wt < threshold:
            return True
    return False

def detect_watermark(response):
    buckets = [decode_token(token, partition) for token in response]
    return detect_llm(buckets, P)

In [148]:
# [resp[0] for resp in response]
buckets = [decode_token(token, partition) for token in response]
detect_llm(buckets, P)

28
30
29
24


True

In [97]:
np.array(P[:, :91] @ GF2(recovered)).sum()

24

In [118]:
np.array(P @ GF2(buckets[:100]))

array([1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0,
       1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0,
       1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0], dtype=uint8)

In [119]:
len(buckets)

197

In [34]:
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


In [35]:
out = model(input_token_ids_tensor)

In [120]:
r = P.shape[0]
(0.5 - (1/(r**4)))*r

27.999994305758015

In [37]:
codeword = torch.Tensor(sample_vector(G, eta)).int()


In [38]:
pos = torch.Tensor(1).int().to('cuda:0')
model.partition[codeword[pos]]*out

AttributeError: 'Qwen3Model' object has no attribute 'partition'

In [ ]:
model.partition.to('cuda:0')

In [ ]:
model.partition[]

In [ ]:
codeword[pos]

P is r * n


G is n * g

#### Assumptions


g<n


r<n

#### Claim
Assume P and G are max possible rank, i.e. rank (P) = min(r,n) and rank(G) = min(g,n)

=> rank(G) = g = dim(nullspace(P))


=> rank(P) = r


r+g = dim(columnspace(P)) = n



P * G = [0]

can r be more than n?

why do we have r+g = n?


In [ ]:
[[0,1,1,0,0,0,0]
[1,0,0,1,0,0,0]]

In [ ]:
Bucket 0
The: 0.1
He : 0.05

Bucket 1
I: 0.3
am: 0.55

In [ ]:
p(0) = 0.15
p(1) = 0.85

In [ ]:
x=0

In [ ]:
t=1

In [ ]:
The: 0.0
He : 0.00

Bucket 1
I: 0.3/0.85
am: 0.55/0.85

In [ ]:
x = 0000


0110 : 0.1
0011 : 0.1
1001 : 0.4
1011 : 0.4
t = 1...